# 01 — Explore SUN RGB-D Subset

Sanity-check visualizations for the curated subset produced by `scripts/buildSubset.py`.

Run the setup cell below first — it mounts Drive and clones the repo. No GPU needed.

In [ ]:
# ── Setup: mount Drive + make roomify importable ───────────────────────────
# Run this first. No GPU needed for this notebook.
import sys, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/roomify')
if not REPO_DIR.exists():
    !git clone https://github.com/ben-blake/roomify.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --quiet

sys.path.insert(0, str(REPO_DIR / 'src'))
print('Setup complete.')

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('/content/roomify')
sys.path.insert(0, str(REPO_ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from roomify.dataset import loadManifest, getRecord, listByScene, VALID_SCENE_TYPES

# Resolve manifest path: Drive-backed if mounted, else local dev fallback
DRIVE_MANIFEST = Path('/content/drive/MyDrive/roomify/data/sunrgbd_subset/manifest.csv')
LOCAL_MANIFEST = REPO_ROOT / 'data' / 'manifest.csv'

manifestPath = DRIVE_MANIFEST if DRIVE_MANIFEST.exists() else LOCAL_MANIFEST
print(f'Loading manifest from: {manifestPath}')

df = loadManifest(manifestPath)
print(f'Total records: {len(df)}')
df.head()

In [ ]:
# Scene type distribution
counts = df['sceneType'].value_counts()
print('Scene type distribution:')
print(counts.to_string())
assert set(counts.index) <= VALID_SCENE_TYPES, f'Unexpected scene types: {set(counts.index) - VALID_SCENE_TYPES}'
print('\nAll scene types valid.')

In [ ]:
# Verify depth file exists for every record
missing_depth = []
for _, row in df.iterrows():
    if not Path(row['depthPath']).exists():
        missing_depth.append(row['id'])

if missing_depth:
    print(f'WARNING: {len(missing_depth)} records missing depth file:')
    for rid in missing_depth[:10]:
        print(f'  {rid}')
else:
    print(f'All {len(df)} depth files present.')

In [ ]:
def showRgbDepthPair(record, ax_rgb, ax_depth):
    """Render one RGB + depth pair onto the given axes."""
    rgb = Image.open(record.rgbPath).convert('RGB')
    depth_raw = Image.open(record.depthPath)

    ax_rgb.imshow(rgb)
    ax_rgb.set_title(f'{record.id}\n{record.sceneType}', fontsize=8)
    ax_rgb.axis('off')

    depth_arr = np.array(depth_raw).astype(float)
    if depth_arr.max() > 0:
        depth_arr = depth_arr / depth_arr.max()
    ax_depth.imshow(depth_arr, cmap='plasma')
    ax_depth.set_title('depth', fontsize=8)
    ax_depth.axis('off')


# Show one RGB+depth pair per scene type
SAMPLES_PER_TYPE = 1
fig, axes = plt.subplots(
    len(VALID_SCENE_TYPES), SAMPLES_PER_TYPE * 2,
    figsize=(6 * SAMPLES_PER_TYPE, 3 * len(VALID_SCENE_TYPES))
)

for row_idx, sceneType in enumerate(sorted(VALID_SCENE_TYPES)):
    scene_df = listByScene(df, sceneType)
    if scene_df.empty:
        axes[row_idx, 0].set_title(f'{sceneType}: no samples', fontsize=8)
        axes[row_idx, 0].axis('off')
        axes[row_idx, 1].axis('off')
        continue

    sample_row = scene_df.iloc[0]
    rec = getRecord(df, sample_row['id'])
    showRgbDepthPair(rec, axes[row_idx, 0], axes[row_idx, 1])

plt.suptitle('SUN RGB-D Subset — one RGB + depth pair per scene type', fontsize=10)
plt.tight_layout()
plt.savefig('sunrgbd_sample_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print('Grid saved to sunrgbd_sample_grid.png')

In [ ]:
# Show 3 samples per scene type for a richer contact sheet
CONTACT_SAMPLES = 3
n_cols = CONTACT_SAMPLES * 2  # rgb + depth per sample
fig, axes = plt.subplots(
    len(VALID_SCENE_TYPES), n_cols,
    figsize=(4 * CONTACT_SAMPLES, 2.5 * len(VALID_SCENE_TYPES))
)

for row_idx, sceneType in enumerate(sorted(VALID_SCENE_TYPES)):
    scene_df = listByScene(df, sceneType)
    samples = scene_df.head(CONTACT_SAMPLES)

    for col_offset, (_, sample_row) in enumerate(samples.iterrows()):
        rec = getRecord(df, sample_row['id'])
        rgb_ax = axes[row_idx, col_offset * 2]
        depth_ax = axes[row_idx, col_offset * 2 + 1]
        try:
            showRgbDepthPair(rec, rgb_ax, depth_ax)
        except Exception as exc:
            rgb_ax.set_title(f'Error: {exc}', fontsize=6)
            rgb_ax.axis('off')
            depth_ax.axis('off')

    # Blank out any unused columns
    for col_offset in range(len(samples), CONTACT_SAMPLES):
        axes[row_idx, col_offset * 2].axis('off')
        axes[row_idx, col_offset * 2 + 1].axis('off')

    # Scene type label on the leftmost axis
    axes[row_idx, 0].set_ylabel(sceneType, fontsize=9, rotation=0, labelpad=60, va='center')

plt.suptitle('SUN RGB-D Subset Contact Sheet (3 per scene type)', fontsize=11)
plt.tight_layout()
plt.savefig('sunrgbd_contact_sheet.png', dpi=100, bbox_inches='tight')
plt.show()
print('Contact sheet saved to sunrgbd_contact_sheet.png')

In [ ]:
# Object label statistics
all_labels = []
for labels_str in df['objectLabels'].dropna():
    if labels_str:
        all_labels.extend([l.strip() for l in str(labels_str).split(',') if l.strip()])

from collections import Counter
label_counts = Counter(all_labels)
top_20 = label_counts.most_common(20)

if top_20:
    labels, counts = zip(*top_20)
    plt.figure(figsize=(10, 4))
    plt.bar(labels, counts)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.title('Top 20 object labels in subset')
    plt.tight_layout()
    plt.savefig('sunrgbd_label_freq.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Unique object labels: {len(label_counts)}')
else:
    print('No object labels available (annotation JSON not present in this dataset split).')

In [ ]:
# Summary
print('=== SUN RGB-D Subset Summary ===')
print(f'Total records : {len(df)}')
for sceneType in sorted(VALID_SCENE_TYPES):
    n = len(listByScene(df, sceneType))
    print(f'  {sceneType:<15}: {n}')
print(f'Unique labels  : {len(label_counts) if top_20 else "N/A"}')
print('Exit criterion: manifest loads, depth present, RGB+depth pairs render correctly.')